In [ ]:
import os, re, json, string, warnings
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
from torchvision.transforms.functional import InterpolationMode

import transformers
transformers.logging.set_verbosity_error()

from transformers import (
    AutoTokenizer, AutoModel, AutoProcessor, AutoModelForImageTextToText,
    LlavaForConditionalGeneration,
    Qwen25VLForConditionalGeneration,
    Qwen3VLForConditionalGeneration,
)

from qwenvlutils import process_vision_info

# -----------------------
# 0) CONFIG (edit paths)
# -----------------------
PUZZLEDIR = "data"
LABELFILE = "Rebus Puzzle.xlsx"
OUTDIR = "agg_results"
os.makedirs(OUTDIR, exist_ok=True)

MAXPUZZLES = 1164
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DTYPE_BF16 = torch.bfloat16 if DEVICE == "cuda" else torch.float32
DTYPE_FP16 = torch.float16 if DEVICE == "cuda" else torch.float32

BATCHSIZE = 1

PROMPT_MAIN = """You are given an image that represents a rebus puzzle (a visual word riddle).\n"
    "A rebus puzzle encodes a common English word or phrase using visual layout, repetition, color, position, or size of text and symbols.\n"
    "Do NOT read the image literally.\n"
    "Infer the hidden word or idiomatic expression suggested by the visual arrangement.\n\n"
    "Examples:\n"
    "- The word 'MAN' written three times means 'three men'.\n"
    "- The word 'READ' inside a box means 'read between the lines'.\n"
    "- The word 'YOU' written above 'ME' means 'you over me'.\n"
    "- A red letter 'E' followed by 'GO GO' means 'ready to go'.\n\n"
    "Question: What English word or phrase is represented?\n"
    "Return ONLY the final answer in 1–5 words.\n"
    "Do not explain.
"""


# -----------------------
# 1) Load labels
# -----------------------
labelsdf = pd.read_excel(LABELFILE, header=None).iloc[2:]
labelsdf.columns = ["puzzleid", "answer"]
labelsdf = labelsdf[pd.to_numeric(labelsdf["puzzleid"], errors="coerce").notna()]
labelsdf["puzzleid"] = labelsdf["puzzleid"].astype(int)
LABELS = {int(r.puzzleid): str(r.answer) for _, r in labelsdf.iterrows()}

# -----------------------
# 2) Torch Dataset
# -----------------------
class RebusDataset(Dataset):
    def __init__(self, puzzledir, labels, maxitems=None, load_pil=True):
        self.samples = []
        for pid in sorted(labels.keys()):
            imgpath = os.path.join(puzzledir, f"puzzle{pid}.png")
            if os.path.exists(imgpath):
                self.samples.append((pid, imgpath, labels[pid]))
            if maxitems is not None and len(self.samples) >= maxitems:
                break
        self.load_pil = load_pil

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, imgpath, ans = self.samples[idx]
        if self.load_pil:
            image = Image.open(imgpath).convert("RGB")
            return pid, imgpath, image, ans
        return pid, imgpath, ans

def collate_keep(batch):
    return batch[0]

dataset = RebusDataset(PUZZLEDIR, LABELS, maxitems=MAXPUZZLES, load_pil=True)
loader = DataLoader(dataset, batch_size=BATCHSIZE, shuffle=False, num_workers=0, collate_fn=collate_keep)

# -----------------------
# 3) Text utils 
# -----------------------
def clean_prediction(s):
    if s is None:
        return ""
    if isinstance(s, list):
        s = s[0] if len(s) else ""
    s = str(s).strip()
    s = re.sub(r"final\s*answer\s*:?", "", s, flags=re.IGNORECASE).strip()
    s = s.strip('"').strip("'").strip("`").strip("*").strip()
    return s.lower().strip()

def normalize_answer(s):
    if s is None:
        return ""
    s = str(s).lower().strip()
    s = s.translate(str.maketrans("", "", string.punctuation))
    s = re.sub(r"\s+", " ", s).strip()
    return s

def token_f1(pred, gt):
    pt = set(normalize_answer(pred).split())
    gt = set(normalize_answer(gt).split())
    if not pt or not gt:
        return 0.0
    common = pt & gt
    if not common:
        return 0.0
    precision = len(common) / len(pt)
    recall = len(common) / len(gt)
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

# -----------------------
# 4) InternVL tiling 
# -----------------------
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

def build_transform(input_size=448):
    return T.Compose([
        T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
        T.Resize((input_size, input_size), interpolation=InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def find_closest_aspect_ratio(aspect_ratio, target_ratios, width, height, imagesize):
    best_ratio_diff = float("inf")
    best_ratio = (1, 1)
    area = width * height
    for ratio in target_ratios:
        target_aspect = ratio[0] / ratio[1]
        diff = abs(aspect_ratio - target_aspect)
        if diff < best_ratio_diff:
            best_ratio_diff = diff
            best_ratio = ratio
        elif diff == best_ratio_diff:
            if area > 0.5 * imagesize * imagesize * ratio[0] * ratio[1]:
                best_ratio = ratio
    return best_ratio

def dynamic_preprocess(image, minnum=1, maxnum=12, imagesize=448, use_thumbnail=True):
    orig_w, orig_h = image.size
    aspect_ratio = orig_w / orig_h
    target_ratios = set()
    for n in range(minnum, maxnum + 1):
        for i in range(1, n + 1):
            for j in range(1, n + 1):
                if i * j <= maxnum and i * j >= minnum:
                    target_ratios.add((i, j))
    target_ratios = sorted(target_ratios, key=lambda x: x[0] * x[1])
    best = find_closest_aspect_ratio(aspect_ratio, target_ratios, orig_w, orig_h, imagesize)
    target_w = imagesize * best[0]
    target_h = imagesize * best[1]
    resized = image.resize((target_w, target_h))
    processed = []
    grid_w = target_w // imagesize
    blocks = best[0] * best[1]
    for i in range(blocks):
        box = (
            (i % grid_w) * imagesize,
            (i // grid_w) * imagesize,
            (i % grid_w + 1) * imagesize,
            (i // grid_w + 1) * imagesize,
        )
        processed.append(resized.crop(box))
    if use_thumbnail and len(processed) != 1:
        processed.append(image.resize((imagesize, imagesize)))
    return processed

def internvl_pixel_values_from_path(imgpath, input_size=448, maxnum=12):
    image = Image.open(imgpath).convert("RGB")
    tiles = dynamic_preprocess(image, imagesize=input_size, use_thumbnail=True, maxnum=maxnum)
    transform = build_transform(input_size)
    pixel_values = torch.stack([transform(t) for t in tiles])
    return pixel_values

# -----------------------
# 5) Model runners 
# -----------------------
def run_internvl_instruct_chat(MODELPATH, model_name, input_size=448, max_tiles=12, dtype=DTYPE_BF16):
    tokenizer = AutoTokenizer.from_pretrained(MODELPATH, trust_remote_code=True, use_fast=False)
    model_kwargs = dict(torch_dtype=dtype, low_cpu_mem_usage=True, trust_remote_code=True)
    if DEVICE == "cuda":
        model_kwargs["device_map"] = "auto"
    model = AutoModel.from_pretrained(MODELPATH, **model_kwargs).eval()

    def infer(imgpath, prompt):
        pixel_values = internvl_pixel_values_from_path(imgpath, input_size=input_size, maxnum=max_tiles).to(dtype=dtype)
        try:
            target_device = model.device
        except Exception:
            target_device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
        pixel_values = pixel_values.to(target_device)
        question = f"<image>\n{prompt}"
        gencfg = dict(max_new_tokens=32, do_sample=False, num_beams=1,
                      repetition_penalty=1.1,
                      pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
                      eos_token_id=tokenizer.eos_token_id)
        out = model.chat(tokenizer=tokenizer, pixel_values=pixel_values, question=question,
                         generation_config=gencfg, history=None, return_history=False)
        if isinstance(out, tuple):
            out = out[0]
        out = str(out).strip().splitlines()[0].strip()
        return clean_prediction(out)

    return infer

def run_internvl_hf_image_text_to_text(MODELPATH, model_name, dtype=DTYPE_BF16):
    processor = AutoProcessor.from_pretrained(MODELPATH, trust_remote_code=True)
    model_kwargs = dict(torch_dtype=dtype, low_cpu_mem_usage=True, trust_remote_code=True)
    if DEVICE == "cuda":
        model_kwargs["device_map"] = "auto"
    model = AutoModelForImageTextToText.from_pretrained(MODELPATH, **model_kwargs).eval()

    def infer(imgpath, prompt):
        image = Image.open(imgpath).convert("RGB")
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
        inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                               return_dict=True, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        if "pixel_values" in inputs:
            inputs["pixel_values"] = inputs["pixel_values"].to(dtype=dtype)
        genids = model.generate(**inputs, max_new_tokens=32, do_sample=False, num_beams=1)
        promptlen = inputs["input_ids"].shape[1]
        outtext = processor.decode(genids[0, promptlen:], skip_special_tokens=True).strip()
        outtext = outtext.splitlines()[0].strip()
        return clean_prediction(outtext)

    return infer

def run_llava(MODELPATH, model_name, dtype=DTYPE_FP16):
    model = LlavaForConditionalGeneration.from_pretrained(
        MODELPATH, torch_dtype=dtype, device_map="auto" if DEVICE == "cuda" else None
    ).eval()
    processor = AutoProcessor.from_pretrained(MODELPATH)

    def infer(imgpath, prompt):
        image = Image.open(imgpath).convert("RGB")
        inputs = processor(text=prompt, images=image, return_tensors="pt")
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        outids = model.generate(**inputs, max_new_tokens=12, do_sample=False,
                                temperature=0.0, repetition_penalty=1.1)
        ans = processor.decode(outids[0], skip_special_tokens=True)
        if "Answer:" in ans:
            ans = ans.split("Answer:")[-1].strip()
        return clean_prediction(ans)

    return infer

def run_qwen25_vl(MODELPATH, model_name, dtype=DTYPE_BF16, max_new_tokens=10):
    model = Qwen25VLForConditionalGeneration.from_pretrained(
        MODELPATH, torch_dtype=dtype, device_map="auto" if DEVICE == "cuda" else None, trust_remote_code=True
    ).eval()
    processor = AutoProcessor.from_pretrained(MODELPATH, trust_remote_code=True, use_fast=False)

    def infer(imgpath, prompt):
        image = Image.open(imgpath).convert("RGB")
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(DEVICE)
        outids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        out = processor.batch_decode(outids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
        out = out[0] if isinstance(out, list) else out
        return clean_prediction(out)

    return infer

def run_qwen3_vl(MODELPATH, model_name, dtype=DTYPE_BF16, max_new_tokens=12):
    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODELPATH, torch_dtype=dtype, device_map="auto" if DEVICE == "cuda" else None, trust_remote_code=True
    ).eval()
    processor = AutoProcessor.from_pretrained(MODELPATH, trust_remote_code=True, use_fast=False)

    def infer(imgpath, prompt):
        image = Image.open(imgpath).convert("RGB")
        messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt").to(DEVICE)
        outids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = outids[0, inputs["input_ids"].shape[1]:]
        out = processor.decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
        return clean_prediction(out)

    return infer

# -----------------------
# 6) Register models (edit paths/ids)
# -----------------------
MODELS = [
    dict(
        name="InternVL3.5-30B-A3B-Instruct",
        kind="internvl_chat",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--OpenGVLab--InternVL35-30B-A3B-Instruct/snapshots/83f9a51dbd940c291fb149debee61502f19444d2"),
        prompt=PROMPT_MAIN,
        kwargs=dict(input_size=448, max_tiles=12, dtype=DTYPE_BF16),
    ),
    dict(
        name="InternVL3.5-8B-Instruct",
        kind="internvl_chat",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--OpenGVLab--InternVL35-8B-Instruct/snapshots/27506812aa9914804018996329e895977ee2d0c8"),
        prompt=PROMPT_MAIN,
        kwargs=dict(input_size=448, max_tiles=12, dtype=DTYPE_BF16),
    ),
    dict(
        name="InternVL3.5-4B",
        kind="internvl_chat",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--OpenGVLab--InternVL35-4B/snapshots/481f6e32467eab4e922ccd7fd6cf420441a62331"),
        prompt=PROMPT_MAIN,
        kwargs=dict(input_size=448, max_tiles=12, dtype=DTYPE_BF16),
    ),
    dict(
        name="LLaVA-1.5-7B-HF",
        kind="llava",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--llava-hf--llava-1.5-7b-hf/snapshots/b234b804b114d9e37bb655e11cbbb5f5e971b7a9"),
        prompt=PROMPT_MAIN,
        kwargs=dict(dtype=DTYPE_FP16),
    ),
    dict(
        name="Qwen2.5-VL-3B",
        kind="qwen25",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3"),
        prompt=PROMPT_MAIN,
        kwargs=dict(dtype=DTYPE_BF16, max_new_tokens=10),
    ),
    dict(
        name="Qwen2.5-VL-7B",
        kind="qwen25",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5"),
        prompt=PROMPT_MAIN,
        kwargs=dict(dtype=DTYPE_BF16, max_new_tokens=10),
    ),
    dict(
        name="Qwen2.5-VL-32B",
        kind="qwen25",
        modelpath=os.path.expanduser("~/Qwen2.5-VL-32B-Instruct"),
        prompt=PROMPT_MAIN,
        kwargs=dict(dtype=DTYPE_BF16, max_new_tokens=12),
    ),
    dict(
        name="Qwen3-VL-4B",
        kind="qwen3",
        modelpath="Qwen/Qwen3-VL-4B-Instruct",
        prompt=PROMPT_MAIN,
        kwargs=dict(dtype=DTYPE_BF16, max_new_tokens=12),
    ),
    dict(
        name="Qwen3-VL-8B",
        kind="qwen3",
        modelpath=os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen3-VL-8B-Instruct/snapshots/0c351dd01ed87e9c1b53cbc748cba10e6187ff3b"),
        prompt=PROMPT_MAIN,
        kwargs=dict(dtype=DTYPE_BF16, max_new_tokens=12),
    ),
]

def build_infer_fn(spec):
    if spec["kind"] == "internvl_chat":
        return run_internvl_instruct_chat(spec["modelpath"], spec["name"], **spec["kwargs"])
    if spec["kind"] == "internvl_hf":
        return run_internvl_hf_image_text_to_text(spec["modelpath"], spec["name"], **spec["kwargs"])
    if spec["kind"] == "llava":
        return run_llava(spec["modelpath"], spec["name"], **spec["kwargs"])
    if spec["kind"] == "qwen25":
        return run_qwen25_vl(spec["modelpath"], spec["name"], **spec["kwargs"])
    if spec["kind"] == "qwen3":
        return run_qwen3_vl(spec["modelpath"], spec["name"], **spec["kwargs"])
    raise ValueError(spec["kind"])

# -----------------------
# 7) Eval all models
# -----------------------
all_summary = []

for spec in MODELS:
    model_name = spec["name"]
    outprefix = os.path.join(OUTDIR, re.sub(r"[^a-zA-Z0-9._-]+", "", model_name.lower()))
    infer_fn = build_infer_fn(spec)

    results = []
    exact_cnt = 0
    f1_sum = 0.0

    print("=" * 80)
    print(f"Evaluating {len(dataset)} puzzles with {model_name}")
    print("=" * 80)

    for pid, imgpath, image, gt in tqdm(loader, total=len(dataset)):
        pred = infer_fn(imgpath, spec["prompt"])
        pred = clean_prediction(pred)

        em = (normalize_answer(pred) == normalize_answer(gt))
        f1 = token_f1(pred, gt)

        exact_cnt += int(em)
        f1_sum += f1

        results.append({
            "model": model_name,
            "puzzleid": int(pid),
            "gtanswer": gt,
            "predanswer": pred,
            "exactmatch": bool(em),
            "tokenf1": float(f1),
        })

    df = pd.DataFrame(results)
    df.to_csv(outprefix + ".csv", index=False)
    with open(outprefix + ".jsonl", "w", encoding="utf-8") as f:
        for r in results:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    n = len(results)
    acc = 100.0 * exact_cnt / n if n else 0.0
    avg_f1 = 100.0 * (f1_sum / n) if n else 0.0

    all_summary.append({
        "model": model_name,
        "num": n,
        "exact_acc": acc,
        "avg_token_f1": avg_f1,
        "csv": os.path.basename(outprefix + ".csv"),
        "jsonl": os.path.basename(outprefix + ".jsonl"),
    })

summary_df = pd.DataFrame(all_summary).sort_values("exact_acc", ascending=False)
summary_df.to_csv(os.path.join(OUTDIR, "summary.csv"), index=False)
print(summary_df)
